In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sparse
from lightfm import LightFM
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k
import nmslib

In [ ]:
ratings = pd.read_csv('C:/Users/Lenovo/IDE/SFDS/data/RecSysN2/data/ratings.csv')
books = pd.read_csv('C:/Users/Lenovo/IDE/SFDS/data/RecSysN2/data/books.csv')
tags = pd.read_csv('C:/Users/Lenovo/IDE/SFDS/data/RecSysN2/data/tags_cleaned.csv')
book_tags = pd.read_csv('C:/Users/Lenovo/IDE/SFDS/data/RecSysN2/data/book_tags.csv')

In [ ]:
books

In [ ]:
mapper = dict(zip(books.goodreads_book_id,books.book_id))
tags = tags
book_tags = book_tags[book_tags.tag_id.isin(tags.tag_id)]
book_tags['id'] = book_tags.goodreads_book_id.apply(lambda x: mapper[x])

In [ ]:
ratings_coo = sparse.coo_matrix((ratings.rating,(ratings.user_id,ratings.book_id)))
feature_ratings  = sparse.coo_matrix(([1]*len(book_tags),(book_tags.id,book_tags.tag_id)))

'''python

число потоков процессора (зависит от того, на какой машине запускаете)
NUM_THREADS = 1

число параметров вектора 
NUM_COMPONENTS = 60 

число эпох обучения
NUM_EPOCHS = 10

зерно датчика случайных чисел
RANDOM_STATE = 42

'''

'''python


#Разбиваем датасет на обучающую и тестовую выборки
train, test = random_train_test_split(ratings_coo, test_percentage=0.2, random_state=RANDOM_STATE)

#Создаём модель
model = LightFM(
    learning_rate=0.05, #темп (скорость) обучения
    loss='warp', #loss-функция
    no_components=NUM_COMPONENTS,#размерность вектора признаков
    random_state=RANDOM_STATE #генератор случайных чисел
    
    
)

#Обучаем модель
model = model.fit(
    train, #обучающая выборка
    epochs=NUM_EPOCHS, #количество эпох обучения
    num_threads=NUM_THREADS, #количество потоков процессора
    item_features=feature_ratings #признаки товаров (рейтинги книг)
    ,verbose=True
)


'''

In [ ]:
import pickle
with open('C:/Users/Lenovo/IDE/SFDS/data/RecSysN2/model.pkl', 'rb') as f:
          model = pickle.load(f)

In [ ]:
# Извлекаем эмбеддинги
item_biases,item_embeddings =  model.get_item_representations(features=feature_ratings)

print(item_biases.shape, item_embeddings.shape)
## (10001,) (10001, 60)



In [ ]:
#Создаём граф для поиска
nms_idx = nmslib.init(method='hnsw', space='cosinesimil')
 
#Начинаем добавлять книги в граф
nms_idx.addDataPointBatch(item_embeddings)
nms_idx.createIndex(print_progress=True)